# Mathematical framework and verification

Here we review the mathematical framework and verify and record the precise computations, assumptions, derivations, analytically predictable examples and small executable checks underlying the token-counting experiment (for bookeeping and reproducibility).

**References.** Zhang, Lipton, Li & Smola, *Dive into Deep Learning*, §10.2 (GRU); Cho et al. (2014), *Learning Phrase Representations using RNN Encoder–Decoder for Statistical Machine Translation*; and [PyTorch `nn.GRU` reference](https://docs.pytorch.org/docs/stable/generated/torch.nn.GRU.html).

## Notation conventions and execution contract

| Symbol | Meaning | Shape or range |
|:--|:--|:--|
| $T$ | Number of timesteps in one batch | integer $\geq 1$ for generated data |
| $B$ | Number of sequences in a batch | integer $\geq 1$ |
| $D$ | Input features per token | positive integer |
| $K$ | Number of GRU hidden coordinates | positive integer |
| $X$ | Full input tensor | $\mathbb R^{T\times B\times D}$ |
| $X_t$ | One timestep of inputs | $\mathbb R^{B\times D}$ |
| $H_0,H_t$ | Initial and subsequent states | $\mathbb R^{B\times K}$ |
| $R_t,Z_t,\widetilde H_t$ | Reset gate, update gate, candidate | $\mathbb R^{B\times K}$ |
| $C_{t,b}$ | Prefix count through step $t$ for sequence $b$ | $\{0,\ldots,t\}$ |
| $Y_b=C_{T,b}$ | Terminal target count | $\{0,\ldots,T\}$ |
| $\hat y_b$ | Continuous predicted count | $\mathbb R$ |
| $p$ | Probability a token is one | $[0,1]$ |

We use **time-major, row-oriented batches**.

Computations start at $t=1$; Python iteration starts at index `0`.

**Notation convention:** capital letters denote batched tensors, while lowercase $x_{t,b}$, $c_{t,b}$, $y_b$, and $\hat y_b$ denote values for a single sequence. Matrix multiplication is written for **row vectors**: $H_TW_{\mathrm{out}}^\top+b_{\mathrm{out}}$, not $W_{\mathrm{out}}H_T$.

## 1. Recurrent computation
Baseline model implementation follows Zhang et al.

**Implementation:** `GRUScratch.__init__()` and `GRUScratch.forward()`.

For input dimension $D$ and hidden dimension $K$:

- Inputs: $X\in\mathbb{R}^{T\times B\times D}$.
- Hidden state: $H_t\in\mathbb{R}^{B\times K}$.
- Input weights: $W_{x*}\in\mathbb{R}^{D\times K}$.
- Recurrent weights: $W_{h*}\in\mathbb{R}^{K\times K}$.

The recurrent transition $H_t=F_\theta(X_t,H_{t-1})$ is

$$
\begin{aligned}
Z_t &= \operatorname{sigmoid}(X_tW_{xz}+H_{t-1}W_{hz}+b_z),\\
R_t &= \operatorname{sigmoid}(X_tW_{xr}+H_{t-1}W_{hr}+b_r),\\
\widetilde H_t &= \tanh\!\left(X_tW_{xh}+(R_t\odot H_{t-1})W_{hh}+b_h\right),\\
H_t &= Z_t\odot H_{t-1}+(1-Z_t)\odot\widetilde H_t.
\end{aligned}
$$

Here $\odot$ denotes elementwise multiplication and the activation functions act elementwise. Each gate uses **independent** parameters. In our convention **$Z_t\approx1$ retains the old state**, whereas $Z_t\approx0$ replaces it with the candidate. The reset gate instead determines which components of the previous state enter the *candidate* computation; it does not directly reset the final state.

**Shapes.** For $g\in\{z,r,h\}$, $W_{xg}\in\mathbb R^{D\times K}$, $W_{hg}\in\mathbb R^{K\times K}$ and $b_g\in\mathbb R^K$. Consequently every gate, candidate and updated state has shape $(B,K)$.

**Reference distinction.** Our candidate multiplies **$(R_t\odot H_{t-1})W_{hh}$**. In PyTorch, native `nn.GRU` uses a candidate of the form

$$n_t = \tanh\left(X_t W_{in\_n} + b_{in\_n} + R_t \odot (H_{t-1} W_{hn\_n} + b_{hn\_n})\right)$$

applying the reset gate after the recurrent linear transformation. The two reset placements are not equivalent. Direct numerical equality with native `nn.GRU` is *not* the correctness criterion for this implementation.

### 1.1 Constructor, random initialisation and parameter count

**Computation.** `GRUScratch(D,K,sigma)` registers three parameter triples $(W_{xg},W_{hg},b_g)$, for update, reset and candidate transformations. Each weight element is drawn as $W_{ij}=\sigma\epsilon_{ij}$ with $\epsilon_{ij}\sim\mathcal N(0,1)$, hence $W_{ij}\sim\mathcal N(0,\sigma^2)$. Each bias starts at zero. The default is $\sigma=0.01$.

**Code correspondence (`gru_vanilla.py`, lines 20–33):**

```python
return nn.Parameter(torch.randn(*shape) * sigma)
# Each of three calls to triple() creates (D,K) weights, (K,K) weights, (K,) bias.
```

**Parameter count derivation.** One triple contains $DK+K^2+K$ trainable scalars, so

$$N_{\mathrm{GRU}}=3(DK+K^2+K).$$

For $D=3,K=5$: $N_{\mathrm{GRU}}=3(15+25+5)=135$. For the counting architecture's $D=1,K=16$: $N_{\mathrm{GRU}}=3(16+256+16)=864$.

**Assumptions and limitations.** `sigma` is the **standard deviation**, not variance. The code uses the global PyTorch RNG state unless seeded by its caller; initialisation does not establish that $\sigma=0.01$ is optimal for learning. The six weight matrices and three biases are separate trainable tensors.

**Independent edge case.** When $\sigma=0$, all weights must be exactly zero irrespective of the random draws. This is a deterministic implementation check.

In [11]:
import math
import torch

from gru_vanilla import GRUScratch
from counting_experiment import (
    CountingGRUSystem,
    _model_input,
    generate_counting_batch,
    length_batches,
)

# Constructor and registration checks, including a deterministic sigma=0 case.
for D, K, expected in ((3, 5, 135), (1, 16, 864)):
    torch.manual_seed(11)
    gru = GRUScratch(D, K)
    params = dict(gru.named_parameters())
    assert len(params) == 9
    assert sum(p.numel() for p in params.values()) == expected
    assert all(p.requires_grad for p in params.values())
    for gate in ("z", "r", "h"):
        assert params[f"W_x{gate}"].shape == (D, K)
        assert params[f"W_h{gate}"].shape == (K, K)
        assert params[f"b_{gate}"].shape == (K,)
        assert torch.count_nonzero(params[f"b_{gate}"]).item() == 0

zero_gru = GRUScratch(1, 2, sigma=0)
assert all(torch.count_nonzero(p).item() == 0 for p in zero_gru.parameters())
print("PASS — registered shapes, 135/864 parameter counts, trainability, zero biases, sigma=0")

PASS — registered shapes, 135/864 parameter counts, trainability, zero biases, sigma=0


### 1.2 Forward pass: initial state, gate coefficients, and recurrent update

**Computation and shapes.** `forward(inputs, H=None, ...)` consumes $X\in\mathbb R^{T\times B\times D}$; if `H` is omitted it sets $H_0=0\in\mathbb R^{B\times K}$ using the input dtype/device. At each timestep the GRU computes $Z_t$, $R_t$, $\widetilde H_t$, then $H_t$, storing each $H_t$. The ordinary return is `([H_1, ..., H_T], H_T)`.

**Code correspondence (`gru_vanilla.py`, lines 43–71):**

```python
if H is None:
    H = inputs.new_zeros((inputs.shape[1], self.num_hiddens))
for t, X in enumerate(inputs):
    # Compute Z, R, H_tilde from the four equations above.
    H = Z_effective * H + (1 - Z_effective) * H_tilde
    outputs.append(H)
```

**Why the update is an interpolation.** For each coordinate, $h_t=zh_{t-1}+(1-z)\widetilde h_t$, where $0\leq z\leq1$ in exact arithmetic. Hence:

- $z=1$: retain $h_{t-1}$;
- $z=0$: replace with $\widetilde h_t$;
- $z=1/2$: average the two.

**Analytical two-step example.** Take $B=D=K=1$, $H_0=0.4$, $x_1=1$, $x_2=0$. Let gate preactivations be zero, so $z_t=r_t=1/2$, and set $W_{xh}=0.8$, $W_{hh}=0.3$, $b_h=0$. Then

$$\widetilde h_t=\tanh(0.8x_t+0.15h_{t-1}),\qquad h_t=\tfrac12h_{t-1}+\tfrac12\widetilde h_t.$$

Thus

$$h_1=0.2+\tfrac12\tanh(0.86),\qquad h_2=\tfrac12h_1+\tfrac12\tanh(0.15h_1).$$

The test evaluates these scalar expressions with Python's `math.tanh` rather than implementing a second PyTorch GRU.

In [12]:
# Evaluate scalar, two-step recurrence
gru = GRUScratch(1, 1, sigma=0).double()
with torch.no_grad():
    gru.W_xh.fill_(0.8)
    gru.W_hh.fill_(0.3)

sequence = torch.tensor([1.0, 0.0], dtype=torch.float64).reshape(2, 1, 1)
h0 = torch.tensor([[0.4]], dtype=torch.float64)
outputs, final = gru(sequence, h0)

expected_h1 = 0.5 * 0.4 + 0.5 * math.tanh(0.8 + 0.5 * 0.4 * 0.3)
expected_h2 = 0.5 * expected_h1 + 0.5 * math.tanh(0.5 * expected_h1 * 0.3)
torch.testing.assert_close(outputs[0][0, 0], torch.tensor(expected_h1, dtype=torch.float64), atol=1e-12, rtol=0)
torch.testing.assert_close(final[0, 0], torch.tensor(expected_h2, dtype=torch.float64), atol=1e-12, rtol=0)
assert torch.equal(outputs[-1], final)
print(f"PASS — h1={expected_h1:.8f}, h2={expected_h2:.8f} (scalar analytical reference)")

PASS — h1=0.54812884, h2=0.31508170 (scalar analytical reference)


### 1.3 Reset-gate placement: a distinguishable analytical example

**Question.** Does the reset gate modify the previous state **before** recurrent matrix multiplication?

Choose one row state $H_0=[0.2,-0.4]$, reset gate $R=[0.25,0.75]$, and recurrent matrix

$$W_{hh}=\begin{bmatrix}0&1\\2&0\end{bmatrix}.$$

The intended recurrent candidate preactivation is

$$ (R\odot H_0)W_{hh}=[0.05,-0.30]W_{hh}=\boxed{[-0.60,0.05]}. $$

Resetting *after* multiplication would instead produce

$$R\odot(H_0W_{hh})=[0.25,0.75]\odot[-0.8,0.2]=[-0.2,0.15].$$

Clearly these differ. Setting all other weights and candidate bias to zero and $Z=0.5$, the independently predicted next state is $\frac12H_0+\frac12\tanh([-0.6,0.05])$.

In [13]:
# Check one-step hand-computable example distinguishes reset-before vs reset-after
gru = GRUScratch(1, 2, sigma=0).double()
with torch.no_grad():
    gru.b_r.copy_(torch.tensor([math.log(1/3), math.log(3)], dtype=torch.float64))
    gru.W_hh.copy_(torch.tensor([[0.0, 1.0], [2.0, 0.0]], dtype=torch.float64))

h0 = torch.tensor([[0.2, -0.4]], dtype=torch.float64)
inputs = torch.zeros((1, 1, 1), dtype=torch.float64)
_, observed, gates = gru(inputs, h0, record_gates=True)
expected = torch.tensor([
    0.5 * 0.2 + 0.5 * math.tanh(-0.6),
    0.5 * (-0.4) + 0.5 * math.tanh(0.05),
], dtype=torch.float64).reshape(1, 2)
torch.testing.assert_close(observed, expected, atol=1e-12, rtol=0)
torch.testing.assert_close(gates["r"][0, 0], torch.tensor([0.25, 0.75], dtype=torch.float64), atol=1e-12, rtol=0)
print("PASS — reset before matrix candidate and final state match analytical values")

PASS — reset before matrix candidate and final state match analytical values


### 1.4 Diagnostics, interventions and gradient propagation

**Computation.** `record_gates=True` returns detached $(T,B,K)$ arrays `z_computed`, `z_effective`, `r`. `z_override(X,H,Z,t)` can replace the **effective** update gate, leaving the computed gate available for comparison. With no intervention, $Z_t^{\mathrm{effective}}=Z_t^{\mathrm{computed}}$.

**Assumptions.** The override must return a tensor of shape $(B,K)$ and must not mutate its inputs. Shape is checked, but admissible numerical range, dtype and device are not fully validated; a replacement outside $[0,1]$ no longer has the usual convex-interpolation interpretation. Recorded diagnostics are detached; **the forward states and predictions are not detached** during ordinary differentiable execution.

**Analytical interventions.** If $Z_t^{\mathrm{effective}}\equiv1$, then $H_t=H_{t-1}$ for all $t$, regardless of the candidate. An identity override must agree with the ordinary run.

**Bounded-state invariant.** If $H_0\in[-1,1]^{B\times K}$ and gates have values in $[0,1]$, then $\widetilde H_t\in[-1,1]^{B\times K}$ (by tanh) and each $H_t$ is a coordinatewise convex combination, giving by induction the form $H_t\in[-1,1]^{B\times K}$.

In [14]:
# Check shape, override, detachment and empty-sequence tests.
gru = GRUScratch(1, 2, sigma=0).double()
inputs = torch.tensor([1.0, 0.0, 1.0]).reshape(3, 1, 1).double()
h0 = torch.tensor([[0.4, -0.2]], dtype=torch.float64)

normal_states, normal_final, normal_gates = gru(inputs, h0, record_gates=True)
identity_final = gru(inputs, h0, z_override=lambda X, H, Z, t: Z)[1]
assert torch.equal(normal_final, identity_final)
assert len(normal_states) == 3 and normal_final.shape == (1, 2)
assert normal_states[-1].requires_grad
for values in normal_gates.values():
    assert values.shape == (3, 1, 2) and not values.requires_grad
assert torch.equal(normal_gates["z_computed"], normal_gates["z_effective"])

_, preserved, forced_gates = gru(
    inputs, h0, record_gates=True,
    z_override=lambda X, H, Z, t: torch.ones_like(Z),)

assert torch.equal(preserved, h0)
assert torch.all(forced_gates["z_effective"] == 1)
empty_states, empty_final = gru(inputs[:0], h0)
assert empty_states == [] and torch.equal(empty_final, h0)
print("PASS — trajectories, detached gate records, identity/retention overrides, empty sequence")

PASS — trajectories, detached gate records, identity/retention overrides, empty sequence


In [15]:
# Central finite difference checks a nontrivial two-step parameter gradient
gru = GRUScratch(1, 1, sigma=0).double()
with torch.no_grad():
    gru.W_xh.fill_(0.8)
    gru.W_hh.fill_(0.3)
inputs = torch.tensor([1.0, 0.0], dtype=torch.float64).reshape(2, 1, 1)
h0 = torch.tensor([[0.4]], dtype=torch.float64)

gru.zero_grad()
gru(inputs, h0)[1][0, 0].backward()
autograd_grad = gru.W_xh.grad.item()

eps = 1e-6
original_weight = gru.W_xh.item()
with torch.no_grad():
    gru.W_xh.fill_(original_weight + eps)
    plus = gru(inputs, h0)[1][0, 0].item()
    gru.W_xh.fill_(original_weight - eps)
    minus = gru(inputs, h0)[1][0, 0].item()
    gru.W_xh.fill_(original_weight)
finite_difference = (plus - minus) / (2 * eps)
assert abs(autograd_grad - finite_difference) < 1e-7
print(f"PASS — parameter gradient: autograd={autograd_grad:.10f}, finite difference={finite_difference:.10f}")

PASS — parameter gradient: autograd=0.1479972367, finite difference=0.1479972367


## 2. CountingGRUSystem: recurrent representation and scalar readout

**Code correspondence (`counting_experiment.py`, lines 14–27):**

```python
self.gru = GRUScratch(1, num_hiddens, sigma)
self.readout = nn.Linear(num_hiddens, 1)
# In forward:
outputs, H_final = self.gru(X, H, record_gates, z_override)[:2]
prediction = self.readout(H_final).squeeze(-1)
```

The wrapper instantiates `GRUScratch(D=1, K, sigma)` and `nn.Linear(K,1)`. Starting from $H_0=0$ unless specified, it computes

$$H_t=F_\theta(X_t,H_{t-1}),\qquad \hat Y=H_TW_{\mathrm{out}}^\top+b_{\mathrm{out}}\in\mathbb R^{B\times1}.$$

A final `.squeeze(-1)` gives shape $(B,)$, retaining that dimension even for $B=1$. The readout weights have shape $(1,K)$ and bias $(1,)$. They use **PyTorch `nn.Linear` default initialisation**, not the GRU's `sigma` rule.

**Parameter count.** With $D=1,K=16$, the GRU has $864$ scalars and the affine readout has $16+1=17$, totaling **881 trainable scalars**.

**Why the prediction is continuous.** The supervised count is integer-valued, but the affine prediction is real-valued. Under eventual MSE training, a prediction $2.8$ for target $3$ incurs loss $(2.8-3)^2=0.04$. Rounding can be used to measure exact-count accuracy **at evaluation**, but inserting `round()` in ordinary backpropagation would give zero derivative almost everywhere. Integer targets do not require integer-valued internal states.

**Representation is not prescribed.** The wrapper does **not** implement $c_t=c_{t-1}+x_t$. If the model predicts counts accurately, the final hidden state must contain information accessible to its linear readout, but no single coordinate is required to equal the cumulative count. For instance, $h_{T,1}=0.1Y$ and readout weight $10$ could represent finite counts. This is a *possibility*, not a claim about what our trained GRU actually learns.

**Diagnostic mode.** `return_states=True` stacks detached $(T,B,K)$ states; `record_gates=True` adds detached gate trajectories. The returned prediction retains autograd connectivity. `_model_input(model,X)` aligns tensor dtype/device to the first model parameter; this is execution plumbing, not a change to the statistical task.

**Analytical example.** With a zero-parameter GRU and $H_0=[0.4,-0.2]$, $Z=R=0.5$, $\widetilde H_t=0$, hence $H_2=[0.1,-0.05]$. Readout $W_{\mathrm{out}}=[2,-3]$, $b_{\mathrm{out}}=0.5$ gives $\hat y=0.1(2)+(-0.05)(-3)+0.5=0.85$.

In [16]:
# Controlled wrapper test: manual hidden-state decay and affine readout
model = CountingGRUSystem(num_hiddens=2, sigma=0).double()
with torch.no_grad():
    for parameter in model.gru.parameters():
        parameter.zero_()
    model.readout.weight.copy_(torch.tensor([[2.0, -3.0]], dtype=torch.float64))
    model.readout.bias.fill_(0.5)

inputs = torch.tensor([1.0, 0.0]).reshape(2, 1, 1).double()
h0 = torch.tensor([[0.4, -0.2]], dtype=torch.float64)
prediction, info = model(inputs, H=h0, return_states=True, record_gates=True)
expected_states = torch.tensor([[[0.2, -0.1]], [[0.1, -0.05]]], dtype=torch.float64)
torch.testing.assert_close(info["states"], expected_states, atol=1e-12, rtol=0)
torch.testing.assert_close(prediction, torch.tensor([0.85], dtype=torch.float64), atol=1e-12, rtol=0)
assert prediction.shape == (1,) and prediction.requires_grad
assert not info["states"].requires_grad and not info["H_final"].requires_grad
assert all(not gate.requires_grad for gate in info["gates"].values())

model.zero_grad()
prediction.sum().backward()
torch.testing.assert_close(model.readout.weight.grad, expected_states[-1], atol=1e-12, rtol=0)
torch.testing.assert_close(model.readout.bias.grad, torch.ones(1, dtype=torch.float64))

standard_model = CountingGRUSystem(16)
assert sum(p.numel() for p in standard_model.parameters()) == 881
assert standard_model(torch.zeros(2, 3, 1)).shape == (3,)
assert _model_input(model, torch.ones(1, dtype=torch.float32)).dtype == torch.float64
print("PASS — counting wrapper, readout=0.85, gradient and detached diagnostics, 881 parameters")

PASS — counting wrapper, readout=0.85, gradient and detached diagnostics, 881 parameters


## 3. `generate_counting_batch`: defining the statistical problem

### 3.1 Mathematical sampling model and target definition

For a fixed length $T$, batch size $B$, and token probability $p\in[0,1]$, define for $t=1,\ldots,T$ and $b=1,\ldots,B$:

$$x_{t,b}\overset{\mathrm{iid}}\sim\operatorname{Bernoulli}(p),\qquad
\Pr(x_{t,b}=1)=p,\quad \Pr(x_{t,b}=0)=1-p.$$

Prefix counts and terminal targets are **deterministic** given the sampled tokens:

$$C_{t,b}=\sum_{s=1}^tX_{s,b},\qquad Y_b=C_{T,b}=\sum_{t=1}^{T}x_{t,b}.$$

**Analytical example:** tokens $[0,1,0,1,1]$ produce count prefixes $[0,1,1,2,3]$ and final target $3$.

**Shapes:** `X: (T,B,1)` (float32 binary values), `y_steps: (T,B)` and `target: (B,)`. The function returns `(X, y_steps[-1], y_steps)`.

### 3.2 Sampling construction, line by line

**Code correspondence (`counting_experiment.py`, lines 56–63):**

```python
X = (torch.rand(T, batch_size, 1, generator=generator) < p).to(
    device=device, dtype=torch.float32
)
y_steps = X[..., 0].cumsum(0)
return X, y_steps[-1], y_steps
```

1. Require $T\geq1$, $B\geq1$ and $0\leq p\leq1$; the code assumes integer dimensions.
2. Draw $U_{t,b}\sim\operatorname{Uniform}[0,1)$ with `torch.rand(T,B,1, generator=generator)`.
3. Threshold: $x_{t,b}=\mathbf1[U_{t,b}<p]$. Under ideal uniform draws, $\Pr(x_{t,b}=1)=p$.
4. Convert the Boolean result to float32 and move it to the requested device. The optional generator is a **CPU generator**; sampling occurs on CPU before transfer.
5. Compute `X[..., 0].cumsum(0)`: remove the singleton feature dimension and sum **over time**, not across examples.
6. Return the final row as the terminal count. Integer values up to our tested horizons are exactly representable in float32.

The model later receives `X`, not the target. The prefix counts are provided for diagnostics; the current counting training objective uses only the terminal target.

### 3.3 Distribution of the target (derivation)

Since $Y_b$ is a sum of $T$ iid Bernoulli variables, **conditional on length**,

$$Y\mid T\sim\operatorname{Binomial}(T,p),$$

$$\Pr(Y=k\mid T)=\binom Tk p^k(1-p)^{T-k},\qquad k=0,\ldots,T.$$

Linearity of expectation gives

$$\mathbb E[Y\mid T]=\sum_{t=1}^T\mathbb E[X_t]=\boxed{pT}.$$

Independence gives

$$\operatorname{Var}(Y\mid T)=\sum_{t=1}^T\operatorname{Var}(X_t)=\boxed{Tp(1-p)}.$$

**For $p=0.5$ and $T=10$:** the mean count is $5$ and the variance is $2.5$. Every observed sequence has an **exact** integer target; $pT$ is only the conditional *mean*, not the correct count for an arbitrary sample.

In [17]:
# Boundary cases plus independent Python-integer reconstruction of targets.
for p in (0.0, 1.0):
    X, terminal, prefixes = generate_counting_batch(4, batch_size=3, p=p)
    assert X.shape == (4, 3, 1)
    assert torch.all(X == p)
    assert torch.all(terminal == 4 * p)
    for t in range(4):
        assert torch.all(prefixes[t] == (t + 1) * p)

X, terminal, prefixes = generate_counting_batch(
    7, batch_size=4, p=0.5,
    generator=torch.Generator().manual_seed(42),
)
assert X.shape == (7, 4, 1)
assert terminal.shape == (4,) and prefixes.shape == (7, 4)
assert X.dtype == torch.float32
for b in range(4):
    running = 0
    for t in range(7):
        token = int(X[t, b, 0].item())
        assert token in (0, 1)
        running += token
        assert prefixes[t, b].item() == running
    assert terminal[b].item() == running
print("PASS — exact boundary cases and Python-integer prefix/final target reconstruction")

PASS — exact boundary cases and Python-integer prefix/final target reconstruction


In [18]:
# Statistical sanity check
T, B, p = 10, 20_000, 0.5
_, targets, _ = generate_counting_batch(
    T, batch_size=B, p=p,
    generator=torch.Generator().manual_seed(2026),
)
theoretical_mean = T * p
theoretical_variance = T * p * (1 - p)
observed_mean = targets.mean().item()
observed_variance = targets.var(unbiased=False).item()
mean_standard_error = math.sqrt(theoretical_variance / B)
assert abs(observed_mean - theoretical_mean) < 5 * mean_standard_error
assert abs(observed_variance - theoretical_variance) < 0.15
print(f"PASS — Binomial(10, 0.5) sample: mean={observed_mean:.4f} (theory 5.0000), "
      f"variance={observed_variance:.4f} (theory 2.5000)")

PASS — Binomial(10, 0.5) sample: mean=5.0082 (theory 5.0000), variance=2.4788 (theory 2.5000)


### 3.4 A length-only predictor is statistically legitimate—but not exact counting

The prediction $\hat Y=pT$ uses length, not the realised token values. Under squared loss, it is the optimal predictor among functions that observe **only $T$**:

$$\mathbb E[(Y-pT)^2\mid T]=\operatorname{Var}(Y\mid T)=Tp(1-p).$$

For example, $p=0.5$, $T=10$ gives mean-squared error $2.5$ *in expectation*, while the exact oracle $\sum_tX_t$ has error zero for every sequence.

**Conditional illustration for the later training review.** *If* lengths are uniformly sampled from $\{5,\ldots,20\}$ and $p=0.5$, then

$$\mathbb E[T]=12.5,\quad\operatorname{Var}(T)=21.25,\quad
\mathbb E[Y]=p\mathbb E[T]=6.25.$$

The expected MSE of the length-only predictor is

$$\mathbb E[Tp(1-p)]=3.125.$$

The expected MSE of the best **constant** predictor $\hat Y=6.25$ follows the law of total variance:

$$\operatorname{Var}(Y)=\mathbb E[\operatorname{Var}(Y\mid T)]+\operatorname{Var}(\mathbb E[Y\mid T])=3.125+0.25(21.25)=8.4375.$$

These are theoretical baselines **under the stated length distribution**, not observed model results and not a training-protocol verification. The actual distribution of $T$ during optimisation will be reviewed when we reach `train_counting()`.

In [19]:
# Pure arithmetic check of the conditional illustrative baseline values
lengths = list(range(5, 21))
mean_T = sum(lengths) / len(lengths)
var_T = sum((T - mean_T) ** 2 for T in lengths) / len(lengths)
length_only_mse = sum(T / 4 for T in lengths) / len(lengths)
constant_mse = length_only_mse + var_T / 4
assert (mean_T, var_T, length_only_mse, constant_mse) == (12.5, 21.25, 3.125, 8.4375)
print("PASS — conditional theory: E[T]=12.5, Var(T)=21.25, length-only MSE=3.125, constant MSE=8.4375")

PASS — conditional theory: E[T]=12.5, Var(T)=21.25, length-only MSE=3.125, constant MSE=8.4375


**Notes for §3.** The generator implements Bernoulli threshold sampling and **exact target construction**. The task is statistically well defined for a fixed $T,B,p$. The count–length correlation $\mathbb E[Y\mid T]=pT$ is a property of the distribution; it is not label leakage or a model implementation defect. A statistical sample check supports the intended distribution but cannot prove independence or randomness.

## 4. `length_batches`: fixed-length evaluation collections

Given an **ordered** collection of distinct lengths $\mathcal T=(T_1,\ldots,T_m)$, it constructs

$$\mathcal B=\{T\mapsto X^{(T)}:T\in\mathcal T\},\qquad
X^{(T)}\in\{0,1\}^{T\times B\times1},\qquad
x^{(T)}_{t,b}\sim\operatorname{Bernoulli}(p).$$

**Code correspondence (`counting_experiment.py`, lines 66–70):**

```python
generator = torch.Generator().manual_seed(seed)
return {
    T: generate_counting_batch(T, batch_size, p, device, generator)[0]
    for T in lengths
}
```

**Implementation:** `torch.Generator().manual_seed(seed)` creates one local CPU RNG; a dictionary comprehension calls `generate_counting_batch(...)[0]` **once per listed length**. The `[0]` keeps only the input tensor; later evaluation functions can recover its true counts as $Y_b^{(T)}=\sum_tx^{(T)}_{t,b}$.

**Example.** With lengths `(5, 10, 20)` and batch size 256, the values have shapes `(5,256,1)`, `(10,256,1)`, `(20,256,1)`.

**Reproducibility assumptions and caveats.** Identical seed, arguments, and **order of lengths** reproduce the same tensors in a fixed runtime. One generator is advanced across lengths, so changing the order can change the samples associated with a given $T$. Duplicate keys overwrite earlier entries in the dictionary; use distinct lengths. This function is **not** a training curriculum and does **not** choose the training distribution over lengths.

**Scientific implication.** At fixed $p$, increasing $T$ changes both the number of recurrent transitions and the count distribution:

$$\mathbb E[Y\mid T]=pT,\qquad \operatorname{Var}(Y\mid T)=Tp(1-p).$$

Therefore, ordinary long-length evaluations confound longer computational horizon with larger expected count. This is not a sampling bug. Later **exact-count controls** may help separate aspects of these effects.

In [20]:
# Shapes, deterministic endpoints, local-seed reproducibility, and ordering caveat.
requested = (5, 10, 20)
for p in (0.0, 1.0):
    batches = length_batches(requested, batch_size=8, p=p, seed=91)
    assert tuple(batches) == requested
    for T, X in batches.items():
        assert X.shape == (T, 8, 1)
        assert torch.all(X == p)

first = length_batches(requested, batch_size=8, p=0.5, seed=91)
repeat = length_batches(requested, batch_size=8, p=0.5, seed=91)
reordered = length_batches(tuple(reversed(requested)), batch_size=8, p=0.5, seed=91)
assert all(torch.equal(first[T], repeat[T]) for T in requested)
assert all(torch.all((first[T] == 0) | (first[T] == 1)) for T in requested)
assert not torch.equal(first[5], reordered[5])  # Same seed; different RNG consumption order.
print("PASS — per-length shapes, boundary tokens, deterministic replay, order-sensitive sampling")

PASS — per-length shapes, boundary tokens, deterministic replay, order-sensitive sampling


## 5. `exact_count_batch`: exact count recurrence generation

**Note** This function generates binary sequences with a fixed, precisely controlled number of ones. This is an important methodological component of the experiment because, unlike ordinary Bernoulli sampling, it allows us to specify both sequence length and target count.

**Maths.** The function constructs $B$ binary sequences, each satisfying two constraints
$$
X^{(b)}\in\{0,1\}^{T}
$$

and
$$
\sum_{t=1}^{T}X_{t,b}=k,
$$

where $k=\texttt{count}$. For example, suppose $ T=8, k=3$, then every generated sequence must contain exactly three ones and five zeros.

The function returns only the input tensor X, with shape $(T,B,1)$.

**Definition.**
Let
$$
\Omega_{T,k}
=
\left\{
x\in\{0,1\}^{T}:
\sum_{t=1}^{T}x_t=k
\right\}.
$$

This is the set of all binary sequences of length $T$ containing exactly $k$ ones.

The number of distinct sequences is

$$
|\Omega_{T,k}|=\binom{T}{k}.
$$

Why? Because a sequence is fully determined by choosing which $k$ of the $T$ positions contain ones.

**Random ordering.**

Under ideal continuous random sampling, the random condition selects every valid arrangement with equal probability:
$$
P(X=x\mid T,k)
=
\frac{1}{\binom Tk},
\quad x\in\Omega_{T,k}.
$$

This is a uniform distribution over all sequences with the specified length and count. There is an important relationship to our earlier Bernoulli task. Suppose
$$
X_t\overset{\mathrm{iid}}{\sim}\operatorname{Bernoulli}(p)
$$

for $0<p<1$. For any particular sequence x containing k ones,
$$
P(X=x)=p^k(1-p)^{T-k}.
$$

The probability that the total count equals $k$ is
$$
P(Y_T=k)=\binom Tk p^k(1-p)^{T-k}.
$$

Therefore,
$$
\begin{aligned}
P(X=x\mid Y_T=k)
&=\frac{P(X=x)}{P(Y_T=k)}\\
&=\frac{p^k(1-p)^{T-k}}
{\binom Tk p^k(1-p)^{T-k}}\\
&=\frac1{\binom Tk}.
\end{aligned}
$$

This is a central statistical finding. The random exact-count distribution is the same as the original iid Bernoulli distribution conditioned on a particular final count. This holds for any $0<p<1$.

However, once we condition on the count, the token positions are no longer statistically independent. For example, if T=4 and exactly three ones must appear, seeing three ones in the first three positions guarantees the final token is zero.

In [21]:
from counting_experiment import exact_count_batch

expected_positions = {
    "front": [0, 1, 2],
    "back": [7, 8, 9],
    "spread": [0, 4, 9],
}

for order, expected in expected_positions.items():
    X = exact_count_batch(T=10, count=3, batch_size=4, order=order)

    assert X.shape == (10, 4, 1)

    for b in range(4):
        positions = torch.nonzero(X[:, b, 0]).flatten().tolist()
        assert positions == expected

# Fixed-count invariant under random positioning
X = exact_count_batch(
    T=20, count=6, batch_size=128, order="random",
    generator=torch.Generator().manual_seed(42),
)

assert all(
    sum(int(X[t, b, 0]) for t in range(20)) == 6
    for b in range(128)
)

print("Exact-count generator checks passed.")

Exact-count generator checks passed.


**Notes for §5.** This is an important part of the methodology hen it comes to the experimental design. By fixing the count, we can compare by holding the true target count constant. Changing length while holding count fixed also changes the frequency and spacing of zero tokens, as well as the rarity of the event under the original Bernoulli training distribution. Therefore, if the GRU gives different predictions to these different lengths, we can establish that the model's prediction is not invariant to these length changes when the true count is fixed.

## 5. `controlled_batches`
This function assembles exact-count sequences into four families of controlled evaluation conditions.

**Source:** `counting_experiment.py`, lines 101–120.

The function constructs **17 evaluation conditions** using `exact_count_batch()`. These are behavioural diagnostics, not training stages.

### i. Mathematical definitions

Let a binary sequence be

$$
x=(x_1,\ldots,x_T)\in\{0,1\}^T,
$$

with sequence length $T$, exact count $k$, and empirical one-frequency $f$:

$$
k=\sum_{t=1}^{T}x_t,
\qquad
f=\frac{k}{T}.
$$

The set of all sequences with fixed length $T$ and count $k$ is

$$
\Omega_{T,k}
=
\left\{
x\in\{0,1\}^{T}:
\sum_{t=1}^{T}x_t=k
\right\},
$$

with cardinality

$$
|\Omega_{T,k}|=\binom{T}{k}.
$$

For `order="random"`, the intended sampling distribution is

$$
X\sim\operatorname{Uniform}(\Omega_{T,k}).
$$

This is equivalent to iid Bernoulli sampling conditioned on the final count, for $0<p<1$:

$$
P(X=x\mid Y_T=k,T)
=
\frac{1}{\binom{T}{k}},
\qquad x\in\Omega_{T,k}.
$$

Conditional on the count, token positions are no longer independent.

Importantly, **empirical frequency $f=k/T$ is not the Bernoulli**.

### ii. Controlled evaluation families

| Family | Held fixed | Varied | Conditions |
|---|---|---|---|
| Matched count | $k=6$ | $T$ | $T\in\{10,20,40,80\}$ |
| Matched length | $T=20$ | $k$ | $k\in\{0,2,6,10,20\}$ |
| Matched frequency | $f=0.25$ | $T$ and $k$ | $T\in\{8,20,40,80\}$, $k=T/4$ |
| Matched multiset | $T=40$, $k=6$ | Token arrangement | Front, back, spread, random |

All conditions use the same binary alphabet, $\{0,1\}$.

#### A. Matched count

$$
X^{(T)}\sim\operatorname{Uniform}(\Omega_{T,6}),
\qquad
T\in\{10,20,40,80\}.
$$

Every sequence has the same target:

$$
Y=6.
$$

**Question:** Does the model maintain the correct count prediction as the sequence becomes longer?

Holding $k$ fixed does not isolate length alone. The number of zeros, empirical frequency, and distribution of gaps between ones also change:

| $T$ | $k$ | Zeros | $f=k/T$ |
|---|---|---|---|
| 10 | 6 | 4 | 0.60 |
| 20 | 6 | 14 | 0.30 |
| 40 | 6 | 34 | 0.15 |
| 80 | 6 | 74 | 0.075 |

A prediction difference establishes sensitivity to these changed sequence conditions, but does not by itself identify the responsible recurrent mechanism.

#### B. Matched length

$$
X^{(k)}\sim\operatorname{Uniform}(\Omega_{20,k}),
\qquad
k\in\{0,2,6,10,20\}.
$$

The processing horizon is fixed at $T=20$, while the target changes:

$$
Y=k.
$$

**Question:** Can the GRU accurately distinguish different count magnitudes at a fixed sequence length?

Because

$$
f=\frac{k}{20},
$$

changing $k$ also changes empirical frequency.

The extreme cases $k=0$ and $k=20$ are within the support of the Bernoulli(0.5) training distribution, but are exceptionally rare:

$$
P(Y=0\mid T=20)
=
P(Y=20\mid T=20)
=
2^{-20}.
$$

Thus, a length within the training range does not necessarily imply a typical token composition.

#### C. Matched frequency

$$
X^{(T)}
\sim\operatorname{Uniform}(\Omega_{T,T/4}),
\qquad
T\in\{8,20,40,80\}.
$$

The empirical one-frequency is fixed:

$$
f=\frac{k}{T}=0.25.
$$

| $T$ | $k$ | $f$ |
|---|---|---|
| 8 | 2 | 0.25 |
| 20 | 5 | 0.25 |
| 40 | 10 | 0.25 |
| 80 | 20 | 0.25 |

**Question:** How does the model behave when sequence length and required count increase together at constant empirical frequency?

These sequences are **not iid Bernoulli(0.25)** samples. They contain exactly $T/4$ ones, so their token positions are dependent.

This condition also differs from the expected empirical frequency of the training distribution, where $p=0.5$.

#### D. Matched multiset

All conditions satisfy

$$
T=40,\qquad k=6,\qquad f=0.15.
$$

The same multiset of six ones and 34 zeros is arranged in four ways:

- **Front:** ones occupy the first six positions.
- **Back:** ones occupy the last six positions.
- **Spread:** ones are distributed approximately evenly.
- **Random:** six distinct positions are sampled uniformly under the intended sampling model.

For `spread`, the zero-based positions are

$$
(0,8,16,23,31,39).
$$

**Question:** Is the learned prediction invariant to rearranging tokens when the correct count is unchanged?

Exact counting is permutation-invariant:

$$
\sum_{t=1}^{T}x_t
=
\sum_{t=1}^{T}x_{\pi(t)}
$$

for any permutation $\pi$.

An exact counting predictor therefore satisfies

$$
g(x_1,\ldots,x_T)
=
g(x_{\pi(1)},\ldots,x_{\pi(T)}).
$$

Prediction differences across arrangements establish a failure of permutation invariance, but do not identify its internal cause.

### iii. Mathematical properties of an exact counter

The controlled conditions assess several necessary behavioural properties of exact counting.

**Count accuracy**

$$
g(x_{1:T})=\sum_{t=1}^{T}x_t.
$$

**Zero-extension invariance**

For every $m\geq0$,

$$
g(x_1,\ldots,x_T,\underbrace{0,\ldots,0}_{m})
=
g(x_1,\ldots,x_T).
$$

**Permutation invariance**

$$
g(x_{1:T})=g(x_{\pi(1):\pi(T)}).
$$

These are properties of an exact counting function, not properties guaranteed by a trained GRU.

The matched count family is relevant to count preservation at different lengths, but is not a paired zero-extension experiment. A direct zero-extension test requires preserving the original prefix and appending zeros to that same sequence.

Likewise, the matched multiset family tests selected arrangements; it does not exhaustively verify permutation invariance over all possible arrangements.

### iv. Important assumptions and methodological findings

**Relationship between length, count and frequency**

The quantities satisfy

$$
\boxed{k=fT}.
$$

They cannot all vary independently. Holding any two fixed determines the third. Our four families are complementary comparisons rather than independently manipulated causal factors.

**Overlapping conditions**

Some differently labelled batches share the same mathematical distribution. For example,

- `matched_count/T=20/k=6`
- `matched_length/T=20/k=6`

are separately sampled from $\operatorname{Uniform}(\Omega_{20,6})$.

Likewise, `matched_count/T=40/k=6` and `matched_multiset/order=random` share the same distribution.

These labels describe different planned comparisons, not necessarily different underlying data populations.

**Deterministic repetition**

Under `front`, `back`, and `spread`, every batch column contains an identical sequence. A batch size of 64 therefore does not provide 64 independent positional arrangements.

**ID versus OOD**

The training-length range is $T\in\{5,\ldots,20\}$.

Some controlled sequences fall within that range; others exceed it. Even within the training-length range, exact-count controls can select rare token compositions.

We must distinguish length extrapolation from atypical token composition and larger target magnitudes.

**Reproducibility**

The function uses a locally seeded CPU random-number generator, shared across specifications. Results are reproducible for fixed seeds, ordered specifications, and a compatible execution environment.